In [1]:
import datetime
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import Span, Doc

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

pd.set_option('display.width', 1000)

TODAY = datetime.datetime.today().strftime('%Y-%m-%d')

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-05-30 15:27:24,577 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-05-30 15:27:25,357 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-05-30 15:27:25,680 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


In [2]:
TODAY

'2024-05-30'

In [3]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

In [4]:
labelled_sents1 = load_s3_jsonl(
        BUCKET_NAME,
        s3_file_name="job_quality/prodigy/binary_classifier_labelled_data/20240416/job_sentences_labelled_20240416.jsonl",
        local_file=PROJECT_DIR
        / f"inputs/labelled/job_sentences_labelled_20240509.jsonl",
    )[0][10:]

2024-05-30 15:27:27,026 - dap_job_quality - INFO - File job_quality/prodigy/binary_classifier_labelled_data/20240416/job_sentences_labelled_20240416.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240509.jsonl


In [5]:
len(labelled_sents1) # number of ads originally labelled

54

In [6]:
labelled_sents = get_labelled_job_sentences()

len(labelled_sents) # number of chunks that we labelled (each ad was split into chunks of no more than a set length)

2024-05-30 15:27:29,985 - dap_job_quality - INFO - File job_quality/prodigy/labelled_data/job_sentences_labelled_20240528.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240509.jsonl


309

In [7]:
unique_ids = []

for ad in labelled_sents1:
    if ad['id'] not in unique_ids:
        unique_ids.append(ad['id'])

for ad in labelled_sents:
    if ad['meta']['id'] not in unique_ids:
        unique_ids.append(ad['meta']['id'])
        
len(unique_ids) # number of unique ads that we labelled in second batch

121

In [8]:
def get_total_spans(ad_list):
    span_count = 0
    for ad in ad_list:
        if len(ad['spans'])>0:
            span_count += len(ad['spans'])
    return span_count
    
spans1 = get_total_spans(labelled_sents1)
spans2 = get_total_spans(labelled_sents)

spans1+spans2

686

In [9]:
labelled_data = pdu.get_spans_and_sentences(labelled_sents, chunks=True)
len(labelled_data)

75

In [10]:
labelled_data1 = pdu.get_spans_and_sentences(labelled_sents1, chunks=False)
len(labelled_data1)

46

In [11]:
labelled_df = pd.DataFrame(columns=["span", "sent", "text", "job_id", "chunk"])

for job in labelled_data.keys():
    for chunk in labelled_data[job].keys():
        temp_df = pd.DataFrame(labelled_data[job][chunk])
        temp_df["job_id"] = job
        temp_df["chunk"] = chunk
        labelled_df = pd.concat([labelled_df, temp_df])

In [12]:
for job in labelled_data1.keys():
    temp_df = pd.DataFrame(labelled_data1[job])
    temp_df["job_id"] = job
    temp_df["chunk"] = None
    labelled_df = pd.concat([labelled_df, temp_df])

In [13]:
labelled_df_clean = labelled_df[labelled_df['span']!='']

labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)

len(labelled_df_clean)

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_37613/2831689860.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)


636

In [14]:
labelled_df_clean.head()

,span,sent,text,job_id,chunk,label,sentence
0,Flexibility on working hours,"(Flexibility, on, working, hours, The, success...",Flexibility on working hours The successful ca...,48347582,5,benefit,Flexibility on working hours The successful ca...
1,required to work additional paid time,"(Flexibility, on, working, hours, The, success...",Flexibility on working hours The successful ca...,48347582,5,benefit,Flexibility on working hours The successful ca...
2,out of hours appointments,"(Flexibility, on, working, hours, The, success...",Flexibility on working hours The successful ca...,48347582,5,benefit,Flexibility on working hours The successful ca...
3,This may include evenings and weekends,"(This, may, include, evenings, and, weekends, .)",Flexibility on working hours The successful ca...,48347582,5,benefit,This may include evenings and weekends.
4,Discretionary bonus,"(Discretionary, bonus, .)",Flexibility on working hours The successful ca...,48347582,5,benefit,Discretionary bonus.


In [15]:
edge_cases = labelled_df_clean[~labelled_df_clean.apply(lambda row: row['span'] in row['sentence'], axis=1)]

edge_cases

,span,sent,text,job_id,chunk,label,sentence
2,30 Minutes lunch break,(30),Exciting Opportunity for an Experienced Custom...,47400861,0,benefit,30
4,OTE approx. 45k plus +,"(Uncapped, Overtime, opportunity, +, Call, Out...",Field Service Engineer Mobile Engineer Ele...,43848478,0,benefit,Uncapped Overtime opportunity + Call Out (1 we...
2,if you’re 21+ it’s £9.90ph,"(For, 16, -, 20, year, olds, the, starting, wa...",We’ll keep the overnight positions completely ...,47152592,1,benefit,For 16-20 year olds the starting wage is £7.50...
3,Basic entitlement is 30.0 days (pro rata for h...,"(Basic, entitlement, is, 30.0, days, (, pro, r...",Occupational Therapy Technician Team Leader Se...,45332364,None,benefit,Basic entitlement is 30.0 days (pro rata for h...
3,Hours 37.5 hours per week (Covering 10 00 - 1...,"(Job, Title, , 2nd, Line, IT, Support, Engine...",Job Title 2nd Line IT Support Engineer Locati...,45337254,None,benefit,Job Title 2nd Line IT Support Engineer Locati...
0,You will be given a training program when you...,"(Most, of, the, colleagues, hold, 1st, class, ...",Avanti recruitment are currently working with ...,45338109,None,benefit,Most of the colleagues hold 1st class degrees ...
1,Hours Mon- Fri 9am - 17.30pm,"(Property, Block, Manager, Location, , Bognor...",Property Block Manager Location Bognor Regis ...,45351385,None,benefit,Property Block Manager Location Bognor Regis ...
5,Working hours 37.5 hours per week. Monday - ...,"(A, supportive, working, enviroment, Working, ...","Our client, based in west London, is currently...",45415149,None,benefit,A supportive working enviroment Working hours ...
0,"You can earn up to £31,500 p a including regu...","(Why, work, for, us, ?, )",Are you looking to work for a great employer? ...,45509522,None,benefit,Why work for us?
0,Onsite training facilities.,"(The, Engineering, Team, Leader, will, benefit...",Engineering Team Leader (Moulding Background) ...,45531352,None,benefit,The Engineering Team Leader will benefit from ...


In [16]:
labelled_df_filtered = labelled_df_clean[labelled_df_clean.apply(lambda row: row['span'] in row['sentence'], axis=1)]
len(labelled_df_filtered)

626

In [17]:
def get_negative_example_sentences(labelled_df):
    
    # any sentences already extracted are positive examples
    positive_sentences = labelled_df["sentence"].tolist()
    
    # Find all unique job ads/descs
    unique_job_descs = labelled_df[['job_id','chunk', 'text']].drop_duplicates()
    
    # initialise empty df
    negative_examples_df = pd.DataFrame(columns=['job_id', 'chunk','text', 'sentence'])
    
    for _, row in unique_job_descs.iterrows():
        doc = nlp(row['text'])
        sentences = [sent.text for sent in doc.sents]
        
        for sentence in sentences:
            if sentence not in positive_sentences:
                temp_df = pd.DataFrame({'job_id': [row['job_id']],'chunk': [row['chunk']], 'text': [row['text']], 'sentence': [sentence]})
                negative_examples_df = pd.concat([negative_examples_df, temp_df])
    
    return negative_examples_df

In [18]:
negative_examples_df = get_negative_example_sentences(labelled_df_filtered)
negative_examples_df["label"] = 0
negative_examples_df.head()

,job_id,chunk,text,sentence,label
0,48347582,5,Flexibility on working hours The successful ca...,What will you get in return?,0
0,48347582,6,Private Health care. Company Pension. Life Ass...,As this role requires entry to customers' home...,0
0,42643134,0,My client is a reputable manufacturing product...,My client is a reputable manufacturing product...,0
0,42643134,0,My client is a reputable manufacturing product...,Maintain the posting of purchase invoices to t...,0
0,42643134,0,My client is a reputable manufacturing product...,Compile weekly payment run together with assoc...,0


In [19]:
positive_df = labelled_df_filtered[['job_id', 'chunk','sentence', 'span']]
positive_df_short = positive_df.groupby(['job_id', 'chunk','sentence'])['span'].agg(list).reset_index()
positive_df_short.head()

,job_id,chunk,sentence,span
0,41800102,1,The management offer a fantastic support netwo...,[management offer a fantastic support network ...
1,41800102,3,"Staff Nurse, Registered nurse, RGN, neurologic...","[full time, part time,, Hertfordshire£14.00 - ..."
2,41845015,0,This is a field-based role with an organisatio...,[This is a field-based role]
3,41845015,1,To meet the appropriate level of expertise to ...,[appropriate training will be provided as needed]
4,41848690,0,IT Senior Service Desk Analyst Job Type Tempo...,"[Temporary Duration, Expected to last 3 months..."


In [21]:
len(positive_df_short)

253

In [20]:
len(labelled_df_filtered)

626

In [22]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v4.csv")
lookup.head(20)

,dimension,subcategory,target_phrase,Notes
0,pay and benefits,COMP,pension,NaN
1,pay and benefits,COMP,bonus,NaN
2,pay and benefits,COMP,salary,NaN
3,pay and benefits,COMP,compensation,NaN
4,pay and benefits,COMP,pay,NaN
5,pay and benefits,COMP,per annum,NaN
6,pay and benefits,COMP,overtime,NaN
7,pay and benefits,LEAVE,leave,NaN
8,barriers to access,CARING,parental leave,NaN
9,pay and benefits,LEAVE,holiday,NaN


In [23]:
categories = list(lookup['subcategory'].unique())

In [24]:
lookup['target_phrase'].unique()

array(['pension', 'bonus', 'salary', 'compensation', 'pay', 'per annum',
       'overtime', 'leave', 'parental leave', 'holiday', 'vacation',
       'income protection', 'visa sponsorship', 'flexible working hours',
       'compressed hours', 'flexible working arrangements',
       'flexible working options', 'job share', 'flexible shift patterns',
       'shifts booked in advance', 'remote', 'hybrid', 'part time',
       'full time', 'Monday', 'Tuesday', 'Wednesday', 'Thursday',
       'Friday', 'Weekends', 'Permanent position', 'permanent contract',
       'Temporary position', 'temporary contract', 'training',
       'learning & development', 'career advance', 'career progression',
       'Life insurance', 'Private healthcare', 'Cycle to work',
       'Discounts', 'Travel card ', 'supportive team',
       'supportive culture', 'supportive atmosphere',
       'supportive environment', 'reward', 'recognition', 'trade union',
       'make a difference', 'autonomy', 'sense of purpose',


In [25]:
def match_sentence(sentence, category, lookup=lookup):
    matches = 0
    target_phrases = lookup[lookup['subcategory']==category]['target_phrase'].tolist()
    for target in target_phrases:
        if target.lower() in sentence.lower():
            # matched_dimension = lookup[lookup['target_phrase'] == target]['subcategory'].values[0]
            matches +=1
            
    if matches > 0:
        return 1
    else:
        return 0

In [26]:
for category in categories:
    positive_df_short[category] = positive_df_short['sentence'].apply(lambda x: match_sentence(x, category))

In [27]:
df_with_at_least_one_1 = positive_df_short[positive_df_short[categories].any(axis=1)]

# DataFrame where all of the specified columns are 0
df_with_all_zeros = positive_df_short[positive_df_short[categories].any(axis=1) == False]

In [28]:
df_with_all_zeros

,job_id,chunk,sentence,span,COMP,LEAVE,CARING,SPONSORSHIP,FLEX_HOURS,FLEX_LOC,...,SOCIAL,REWARD,VOICE REPRESENTATION,SENSE OF PURPOSE,AUTONOMY,SHIFT,OVERTIME,DISABILITY,M_HEALTH,HEALTH
2,41845015,0,This is a field-based role with an organisatio...,[This is a field-based role],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,41860758,0,Strong commitment to professional development ...,[Strong commitment to professional development...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,41860758,1,A well-resourced school and a stimulating envi...,[A well-resourced school and a stimulating env...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,41860758,1,"Generous non-contact time and Planning, Prepar...","[Generous non-contact time and Planning, Prepa...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,41860758,1,"They strongly support NPQML, NPQSL and Lead Pr...","[They strongly support NPQML, NPQSL and Lead P...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,48347582,6,Other Benefits including our exclusive Avant d...,[exclusive Avant discount platform],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
244,48382873,0,Building Maintenance Technician Luton £26k - £...,"[£26k - £30k, on the tools instead of on the r...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
245,48382873,0,The main difference here is that because it's ...,[site-based],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
250,48511276,0,They offer an ego and politics free working en...,[They offer an ego and politics free working e...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
print(f'Proportion of sentences matched via string matching: {round(len(df_with_at_least_one_1) /len(positive_df_short), 2)}')

Proportion of sentences matched via string matching: 0.62


In [30]:
len(df_with_at_least_one_1)

158

In [31]:
len(positive_df_short)

253

In [32]:
len(df_with_all_zeros)

95

In [ ]:
target_embeddings = model.encode(lookup['target_phrase'].tolist(), show_progress_bar=True)

In [ ]:
lookup['embeddings'] = target_embeddings.tolist()

In [ ]:
def get_n_most_similar_phrases(input_sentence, 
                                 lookup,
                                 model,
                                 n: int = 3):
    # most_similar_phrases = {}
    
    input_embedding = model.encode(input_sentence)
    
    similarities = [cosine_similarity([input_embedding], [embed])[0][0] for embed in lookup['embeddings'].apply(pd.Series).values]
    
    top_indices = np.argsort(similarities)[::-1][:n]
    
    similar_phrases = lookup.iloc[top_indices]
    similar_phrases['similarity'] = [similarities[i] for i in top_indices]
       
    return similar_phrases[['dimension', 'subcategory', 'target_phrase', 'similarity']]

In [ ]:
df_with_all_zeros['target_phrase'] = ""
df_with_all_zeros['subcategory'] = ""
df_with_all_zeros['similarity'] = 0

for idx, row in df_with_all_zeros.iterrows():
    # Get the most similar phrases for the current sentence
    temp_df = get_n_most_similar_phrases(row['sentence'], lookup, model, n=1)
    if not temp_df.empty:
        # Assign the results directly to the DataFrame using the index
        df_with_all_zeros.at[idx, 'target_phrase'] = temp_df['target_phrase'].iloc[0]
        df_with_all_zeros.at[idx, 'subcategory'] = temp_df['subcategory'].iloc[0]
        df_with_all_zeros.at[idx, 'similarity'] = temp_df['similarity'].iloc[0]


In [ ]:
df_with_all_zeros.head()

In [ ]:
df_with_all_zeros.to_csv(PROJECT_DIR / f"inputs/labelling/{TODAY}_data_to_label_manually_for_categories_all_mini_lm_embeddings.csv")

In [33]:
len(df_with_all_zeros)

95